In [5]:
# ============================================================
# Diabetes Tool - Dataset Harmonization Pipeline
# Stitching PIMA + BRFSS + Sylhet into one training-ready dataset
# ============================================================

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Notebook is in /notebooks/, project root is one level up
PROJECT_ROOT = Path().resolve().parent / 'diabetes-tool'
RAW_DATA = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DATA = PROJECT_ROOT / 'data' / 'processed'

# Load all three datasets for risk scoring
pima = pd.read_csv(RAW_DATA / 'pima' / 'diabetes.csv')
brfss = pd.read_csv(RAW_DATA / 'brfss' / 'diabetes_binary_health_indicators_BRFSS2015.csv')
sylhet = pd.read_csv(RAW_DATA / 'sylhet' / 'diabetes_data_upload.csv')

print("Datasets loaded:")
print(f"  PIMA:   {pima.shape}")
print(f"  BRFSS:  {brfss.shape}")
print(f"  Sylhet: {sylhet.shape}")

Datasets loaded:
  PIMA:   (768, 9)
  BRFSS:  (253680, 22)
  Sylhet: (520, 17)


In [8]:
# ============================================================
# STEP 1: Standardize PIMA Dataset
# ============================================================

pima_clean = pima.copy()

# Fix disguised missing values — replace 0s with NaN in biologically impossible columns
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
pima_clean[zero_cols] = pima_clean[zero_cols].replace(0, np.nan)

# Impute missing values with median (robust to outliers)
imputer = SimpleImputer(strategy='median')
pima_clean[zero_cols] = imputer.fit_transform(pima_clean[zero_cols])

# Rename to common schema
pima_clean = pima_clean.rename(columns={
    'Pregnancies': 'pregnancies',
    'Glucose': 'glucose_fasting',
    'BloodPressure': 'blood_pressure_diastolic',
    'SkinThickness': 'skin_thickness',
    'Insulin': 'insulin',
    'BMI': 'bmi',
    'DiabetesPedigreeFunction': 'diabetes_pedigree',
    'Age': 'age',
    'Outcome': 'diabetes_label'
})

# Add source and available feature flags
pima_clean['source'] = 'pima'
pima_clean['gender'] = np.nan        # all female but not encoded
pima_clean['high_bp'] = np.nan
pima_clean['high_cholesterol'] = np.nan
pima_clean['smoker'] = np.nan
pima_clean['physically_active'] = np.nan
pima_clean['polyuria'] = np.nan
pima_clean['polydipsia'] = np.nan
pima_clean['sudden_weight_loss'] = np.nan

print("PIMA standardized:")
print(f"  Shape: {pima_clean.shape}")
print(f"  Missing values after imputation:")
renamed_zero_cols = ['glucose_fasting', 'blood_pressure_diastolic', 
                     'skin_thickness', 'insulin', 'bmi']
print(pima_clean[renamed_zero_cols].isnull().sum())
print(f"\n  Columns: {pima_clean.columns.tolist()}")

PIMA standardized:
  Shape: (768, 18)
  Missing values after imputation:
glucose_fasting             0
blood_pressure_diastolic    0
skin_thickness              0
insulin                     0
bmi                         0
dtype: int64

  Columns: ['pregnancies', 'glucose_fasting', 'blood_pressure_diastolic', 'skin_thickness', 'insulin', 'bmi', 'diabetes_pedigree', 'age', 'diabetes_label', 'source', 'gender', 'high_bp', 'high_cholesterol', 'smoker', 'physically_active', 'polyuria', 'polydipsia', 'sudden_weight_loss']


In [9]:
# ============================================================
# STEP 2: Standardize BRFSS Dataset
# ============================================================

brfss_clean = brfss.copy()

# Rename to common schema
brfss_clean = brfss_clean.rename(columns={
    'Diabetes_binary': 'diabetes_label',
    'HighBP': 'high_bp',
    'HighChol': 'high_cholesterol',
    'BMI': 'bmi',
    'Smoker': 'smoker',
    'PhysActivity': 'physically_active',
    'Sex': 'gender',
    'Age': 'age_category',     # categorical 1-13, not real age
    'HvyAlcoholConsump': 'heavy_alcohol',
    'Stroke': 'stroke',
    'HeartDiseaseorAttack': 'heart_disease',
    'GenHlth': 'general_health',
    'DiffWalk': 'diff_walking',
    'Education': 'education',
    'Income': 'income'
})

# Age category to approximate midpoint age
age_map = {1:18, 2:23, 3:28, 4:33, 5:38, 6:43, 
           7:48, 8:53, 9:58, 10:63, 11:68, 12:73, 13:78}
brfss_clean['age'] = brfss_clean['age_category'].map(age_map)

# Add missing columns from common schema
brfss_clean['source'] = 'brfss'
brfss_clean['pregnancies'] = np.nan
brfss_clean['glucose_fasting'] = np.nan
brfss_clean['blood_pressure_diastolic'] = np.nan
brfss_clean['skin_thickness'] = np.nan
brfss_clean['insulin'] = np.nan
brfss_clean['diabetes_pedigree'] = np.nan
brfss_clean['polyuria'] = np.nan
brfss_clean['polydipsia'] = np.nan
brfss_clean['sudden_weight_loss'] = np.nan

print("BRFSS standardized:")
print(f"  Shape: {brfss_clean.shape}")
print(f"  Missing values: {brfss_clean.isnull().sum().sum()}")
print(f"  Age range after mapping: {brfss_clean['age'].min()} - {brfss_clean['age'].max()}")
print(f"  Class distribution:")
print(brfss_clean['diabetes_label'].value_counts())

BRFSS standardized:
  Shape: (253680, 33)
  Missing values: 2283120
  Age range after mapping: 18 - 78
  Class distribution:
diabetes_label
0.0    218334
1.0     35346
Name: count, dtype: int64


In [10]:
# ============================================================
# STEP 3: Standardize Sylhet Dataset
# ============================================================

sylhet_clean = sylhet.copy()

# Convert Yes/No to 1/0
binary_cols = ['Polyuria', 'Polydipsia', 'sudden weight loss', 'weakness',
               'Polyphagia', 'Genital thrush', 'visual blurring', 'Itching',
               'Irritability', 'delayed healing', 'partial paresis',
               'muscle stiffness', 'Alopecia', 'Obesity']

for col in binary_cols:
    sylhet_clean[col] = (sylhet_clean[col] == 'Yes').astype(float)

# Gender encoding
sylhet_clean['Gender'] = (sylhet_clean['Gender'] == 'Male').astype(float)

# Target encoding
sylhet_clean['diabetes_label'] = (sylhet_clean['class'] == 'Positive').astype(float)

# Rename to common schema
sylhet_clean = sylhet_clean.rename(columns={
    'Age': 'age',
    'Gender': 'gender',
    'Polyuria': 'polyuria',
    'Polydipsia': 'polydipsia',
    'sudden weight loss': 'sudden_weight_loss',
    'Obesity': 'obesity_symptom'
})

# Add missing columns from common schema
sylhet_clean['source'] = 'sylhet'
sylhet_clean['pregnancies'] = np.nan
sylhet_clean['glucose_fasting'] = np.nan
sylhet_clean['blood_pressure_diastolic'] = np.nan
sylhet_clean['skin_thickness'] = np.nan
sylhet_clean['insulin'] = np.nan
sylhet_clean['bmi'] = np.nan
sylhet_clean['diabetes_pedigree'] = np.nan
sylhet_clean['high_bp'] = np.nan
sylhet_clean['high_cholesterol'] = np.nan
sylhet_clean['smoker'] = np.nan
sylhet_clean['physically_active'] = np.nan

# Drop original class column
sylhet_clean = sylhet_clean.drop(columns=['class'])

print("Sylhet standardized:")
print(f"  Shape: {sylhet_clean.shape}")
print(f"  Missing values: {sylhet_clean.isnull().sum().sum()}")
print(f"  Class distribution:")
print(sylhet_clean['diabetes_label'].value_counts())

Sylhet standardized:
  Shape: (520, 29)
  Missing values: 5720
  Class distribution:
diabetes_label
1.0    320
0.0    200
Name: count, dtype: int64


In [11]:
# ============================================================
# STEP 4: Merge All Datasets into Common Schema
# ============================================================

# Define the common columns we want in the final dataset
common_cols = [
    'age', 'gender', 'bmi', 'glucose_fasting', 
    'blood_pressure_diastolic', 'insulin', 'skin_thickness',
    'diabetes_pedigree', 'pregnancies',
    'high_bp', 'high_cholesterol', 'smoker', 'physically_active',
    'polyuria', 'polydipsia', 'sudden_weight_loss',
    'diabetes_label', 'source'
]

# Select only common columns from each dataset
pima_final = pima_clean[common_cols]
brfss_final = brfss_clean[common_cols]
sylhet_final = sylhet_clean[common_cols]

# Concatenate all three
harmonized = pd.concat([pima_final, brfss_final, sylhet_final], 
                        ignore_index=True)

print("Harmonized dataset:")
print(f"  Shape: {harmonized.shape}")
print(f"\n  Source distribution:")
print(harmonized['source'].value_counts())
print(f"\n  Class distribution:")
print(harmonized['diabetes_label'].value_counts())
print(f"\n  Overall diabetes rate: {harmonized['diabetes_label'].mean()*100:.1f}%")
print(f"\n  Missing values per column:")
print(harmonized.isnull().sum())

Harmonized dataset:
  Shape: (254968, 18)

  Source distribution:
source
brfss     253680
pima         768
sylhet       520
Name: count, dtype: int64

  Class distribution:
diabetes_label
0.0    219034
1.0     35934
Name: count, dtype: int64

  Overall diabetes rate: 14.1%

  Missing values per column:
age                              0
gender                         768
bmi                            520
glucose_fasting             254200
blood_pressure_diastolic    254200
insulin                     254200
skin_thickness              254200
diabetes_pedigree           254200
pregnancies                 254200
high_bp                       1288
high_cholesterol              1288
smoker                        1288
physically_active             1288
polyuria                    254448
polydipsia                  254448
sudden_weight_loss          254448
diabetes_label                   0
source                           0
dtype: int64


In [12]:
# ============================================================
# STEP 5: Final Cleaning and Save
# ============================================================

harmonized_clean = harmonized.copy()

# Flag which features are available per source (for model card documentation)
feature_availability = pd.DataFrame({
    'Feature': common_cols[:-2],  # exclude label and source
    'PIMA': ['Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes',
             'No', 'No', 'No', 'No', 'No', 'No', 'No'],
    'BRFSS': ['Yes', 'Yes', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No',
              'Yes', 'Yes', 'Yes', 'Yes', 'No', 'No', 'No'],
    'Sylhet': ['Yes', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No',
               'No', 'No', 'No', 'No', 'Yes', 'Yes', 'Yes']
})

print("Feature availability across datasets:")
print(feature_availability.to_string(index=False))

# Save harmonized dataset
PROCESSED_DATA = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DATA.mkdir(exist_ok=True)

harmonized_clean.to_csv(PROCESSED_DATA / 'harmonized_diabetes.csv', index=False)

print(f"\nHarmonized dataset saved to data/processed/harmonized_diabetes.csv")
print(f"Final shape: {harmonized_clean.shape}")
print(f"\nReady for model training.")

Feature availability across datasets:
                 Feature PIMA BRFSS Sylhet
                     age  Yes   Yes    Yes
                  gender   No   Yes    Yes
                     bmi  Yes   Yes     No
         glucose_fasting  Yes    No     No
blood_pressure_diastolic  Yes    No     No
                 insulin  Yes    No     No
          skin_thickness  Yes    No     No
       diabetes_pedigree  Yes    No     No
             pregnancies  Yes    No     No
                 high_bp   No   Yes     No
        high_cholesterol   No   Yes     No
                  smoker   No   Yes     No
       physically_active   No   Yes     No
                polyuria   No    No    Yes
              polydipsia   No    No    Yes
      sudden_weight_loss   No    No    Yes

Harmonized dataset saved to data/processed/harmonized_diabetes.csv
Final shape: (254968, 18)

Ready for model training.
